# Social Media Engagement — Predictive Pipeline

## 1. Problem Framing

**Business problem:** The social media coordinator needs a way to *forecast* expected `engagement_rate` from post design choices before publishing.

**Who cares about this:** The social media coordinator (day-to-day post planning) and communications director (channel strategy and expectations).

**Predictive or explanatory?** This notebook is **predictive**. The goal is out-of-sample accuracy and stable performance, not coefficient-level interpretation.

**Feature policy (must match explanatory notebook):** We only use **actionable post-design levers** (platform, timing, format, topic, CTA, boost) and deliberately exclude audience-scale or outcome-adjacent variables (impressions, follower counts, raw engagements) so the model reflects what staff can actually change when composing a post.

**Single source of truth:** Feature engineering comes from `ml_scripts/social_engagement_features.py` via `get_x_y` so explanatory and predictive pipelines cannot diverge.

**Output artifacts:**
- `models/social_engagement_predictor.pkl`
- Updates `backend/AngelsLandingv2.API/SeedData/social_engagement_insights.json` with holdout metrics
- Writes `backend/AngelsLandingv2.API/SeedData/social_engagement_post_predictions.json` with per-post predictions


## 2. Data Acquisition, Preparation & Exploration


In [1]:
import json
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")

_cwd = Path.cwd().resolve()
_root = _cwd.parent  # notebooks live in ml-pipelines/, so parent is the repo root
sys.path.insert(0, str(_root / "ml_scripts"))
from social_engagement_features import CATEGORICAL, get_x_y  # noqa: E402


def resolve_csv_path() -> Path:
    for p in [
        Path("data") / "social_media_posts.csv",
        _root / "ml-pipelines" / "data" / "social_media_posts.csv",
        _root / "backend" / "AngelsLandingv2.API" / "SeedData" / "social_media_posts.csv",
    ]:
        if p.is_file():
            return p
    raise FileNotFoundError("social_media_posts.csv not found. Expected at data/social_media_posts.csv")


df = pd.read_csv(resolve_csv_path())
X, y, post_ids = get_x_y(df)

# Keep this notebook aligned with the explanatory feature policy:
# controllable post-design levers only.
expected_cols = [
    "platform",
    "day_of_week",
    "post_type",
    "media_type",
    "call_to_action_type",
    "content_topic",
    "post_hour",
    "num_hashtags",
    "mentions_count",
    "caption_length",
    "has_call_to_action",
    "is_boosted",
]
assert list(X.columns) == expected_cols, (
    "Feature drift detected. Expected controllable-only columns:\n"
    f"{expected_cols}\n"
    f"but got:\n{list(X.columns)}"
)

print("Rows:", len(X))
print("Feature columns:", list(X.columns))


Rows: 812
Feature columns: ['platform', 'day_of_week', 'post_type', 'media_type', 'call_to_action_type', 'content_topic', 'post_hour', 'num_hashtags', 'mentions_count', 'caption_length', 'has_call_to_action', 'is_boosted']


## 3. Modeling (Predictive) & Model Comparison


In [2]:
# Ch. 11–12: preprocess + compare models on holdout
num_cols = [c for c in X.columns if c not in CATEGORICAL]
preprocess = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
        ("num", StandardScaler(), num_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipelines = {
    "LinearRegression": Pipeline([("prep", preprocess), ("m", LinearRegression())]),
    "DecisionTree": Pipeline([("prep", preprocess), ("m", DecisionTreeRegressor(max_depth=8, min_samples_leaf=3, random_state=42))]),
    "RandomForest": Pipeline([("prep", preprocess), ("m", RandomForestRegressor(n_estimators=160, max_depth=14, min_samples_leaf=3, random_state=42, n_jobs=-1))]),
}

results = []
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({
        "model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R2": r2_score(y_test, pred),
    })
print(pd.DataFrame(results))

baseline_mae = mean_absolute_error(y_test, np.full_like(y_test, y_train.mean()))
print(f"Baseline MAE (predict train mean): {baseline_mae:.5f}")


              model       MAE      RMSE        R2
0  LinearRegression  0.037798  0.046461  0.197240
1      DecisionTree  0.032177  0.043538  0.295074
2      RandomForest  0.027355  0.036865  0.494618
Baseline MAE (predict train mean): 0.04235


## 4. Evaluation, Selection & Deployment Artifacts


In [3]:
# Ch. 17: export RandomForest — holdout metrics first, then refit on full data
rf_pipe = pipelines["RandomForest"]
pt = rf_pipe.predict(X_test)
insights_path = _root / "backend" / "AngelsLandingv2.API" / "SeedData" / "social_engagement_insights.json"
payload = {}
if insights_path.is_file():
    payload = json.loads(insights_path.read_text(encoding="utf-8"))
payload["predictiveMaeHoldout"] = float(mean_absolute_error(y_test, pt))
payload["predictiveR2Holdout"] = float(r2_score(y_test, pt))
payload["predictiveRmseHoldout"] = float(mean_squared_error(y_test, pt) ** 0.5)

rf_pipe.fit(X, y)
pred_all = rf_pipe.predict(X)

models_dir = _root / "models"
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / "social_engagement_predictor.pkl"
joblib.dump(rf_pipe, model_path)
print("Saved:", model_path)

rf = rf_pipe.named_steps["m"]
prep = rf_pipe.named_steps["prep"]
imp = pd.Series(rf.feature_importances_, index=prep.get_feature_names_out()).sort_values(ascending=False)
insights_path.parent.mkdir(parents=True, exist_ok=True)
insights_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Updated insights JSON")

scored_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
pred_rows = [
    {"postId": int(pid), "predictedEngagementRate": float(pr), "engagementScoredAt": scored_at}
    for pid, pr in zip(post_ids.astype(int), pred_all)
]
pred_path = insights_path.parent / "social_engagement_post_predictions.json"
pred_path.write_text(json.dumps(pred_rows, indent=2), encoding="utf-8")
print("Wrote", pred_path)

imp.head(15)


Saved: /Users/zacharywaldrip/Intex2/AngelsLandingv2/models/social_engagement_predictor.pkl
Updated insights JSON
Wrote /Users/zacharywaldrip/Intex2/AngelsLandingv2/backend/AngelsLandingv2.API/SeedData/social_engagement_post_predictions.json


num__post_hour                         0.555749
num__caption_length                    0.071418
cat__platform_TikTok                   0.038917
num__num_hashtags                      0.029387
cat__call_to_action_type_(missing)     0.024321
cat__platform_YouTube                  0.021002
num__mentions_count                    0.018420
num__has_call_to_action                0.018122
cat__platform_Instagram                0.014661
cat__platform_LinkedIn                 0.013265
cat__platform_Twitter                  0.013072
cat__content_topic_Education           0.011454
cat__platform_Facebook                 0.010623
cat__call_to_action_type_ShareStory    0.008782
cat__media_type_Photo                  0.008684
dtype: float64

## 5. Deployment Notes

- **Seed files written for local dev:**
  - `backend/AngelsLandingv2.API/SeedData/social_engagement_insights.json` (adds holdout MAE/RMSE/R² under `predictive*Holdout` keys)
  - `backend/AngelsLandingv2.API/SeedData/social_engagement_post_predictions.json` (per-post predicted engagement rate + timestamp)
- **Production scoring path:** the app should score posts using the same feature engineering (`ml_scripts/social_engagement_features.py`) to guarantee parity with this training notebook.
- **Relationship to explanatory notebook:** explanatory OLS produces interpretable levers/coefficient rows; this predictive model provides forecasts and is evaluated primarily on out-of-sample error metrics.